# 00 · Preparar e conferir a execução no Colab

Esta é a entrada para a versão atual do projeto. O código e os protocolos usam uma única raiz, `execucao/`. Os dados e resultados já existentes são preservados. O arquivo `Iatan_codigo_atual.zip` registra o mesmo commit publicado no GitHub.

1. Execute a preparação abaixo; autorize a montagem do Drive quando solicitada.
2. A preparação confere as dependências, recupera apenas insumos ausentes e atualiza as cópias 00–06 da pasta `Notebooks`.
3. Para refazer o fluxo de 2025, siga **02 → 03 → 04 → 05**. Use 01 somente para nova aquisição da EPA. O bloco **Exp33** do 05 pode ser executado isoladamente.
4. O **06** usa os 15 Parquets de 2011–2025; não exige executar 01–05 novamente. Cada sessão nova tem sua própria célula de preparação.

As escolhas e os resultados dos modelos permanecem nos protocolos e notebooks; esta preparação só organiza arquivos e ambiente. [Aula 08, e-Science](../../configs/phase5_iatan.json).


In [ ]:
# 1. MONTAR O DRIVE E LOCALIZAR A RAIZ COMUM
# Dados e resultados ficam em execucao/; não é necessário executar os outros notebooks
# apenas para carregar o código. IATAN_PROJECT_ROOT permite usar outra localização.
from pathlib import Path
from tempfile import TemporaryDirectory
from zipfile import ZipFile
import importlib.util
import json
import os
import re
import sys
import urllib.request

try:
    from google.colab import drive as _drive
except ImportError:
    _drive = None
if _drive is not None and not Path('/content/drive/MyDrive').is_dir():
    _drive.mount('/content/drive')

_default_delivery = (Path('/content/drive/MyDrive') /
                     'Ciência de Dados - Iatan - Process Mining e Experimentos' /
                     'Iatan 2025 - entrega revisada')
_delivery = Path(os.environ.get('IATAN_DELIVERY_ROOT', str(_default_delivery))).expanduser()
if os.environ.get('IATAN_PROJECT_ROOT'):
    root = Path(os.environ['IATAN_PROJECT_ROOT']).expanduser().resolve()
    if not os.environ.get('IATAN_DELIVERY_ROOT'):
        _delivery = root.parent
elif _drive is not None:
    root = (_delivery / 'execucao').resolve()
else:
    _roots = [Path.cwd(), *Path.cwd().parents]
    root = next((p for p in _roots if (p / 'iatan/phase5.py').is_file()), Path.cwd()).resolve()
    if not os.environ.get('IATAN_DELIVERY_ROOT'):
        _delivery = root.parent

# 2. SINCRONIZAR O CÓDIGO PUBLICADO, PRESERVANDO DATA/ E OUTPUTS/
# O ZIP offline é a mesma distribuição do GitHub. Seu comentário identifica o commit.
_code_zip = _delivery / 'Iatan_codigo_atual.zip'
if _drive is not None and not _code_zip.is_file():
    _delivery.mkdir(parents=True, exist_ok=True)
    _api = 'https://api.github.com/repos/prcgui/co2-cems-process-data-science/commits/main'
    _request = urllib.request.Request(_api, headers={'User-Agent': 'iatan-colab'})
    with urllib.request.urlopen(_request, timeout=60) as _response:
        _commit = json.load(_response)['sha']
    if not re.fullmatch('[0-9a-f]{40}', _commit):
        raise RuntimeError('A versão publicada não pôde ser identificada.')
    _url = 'https://codeload.github.com/prcgui/co2-cems-process-data-science/zip/' + _commit
    with urllib.request.urlopen(_url, timeout=120) as _response:
        _payload = _response.read()
    _temporary_zip = _code_zip.with_suffix('.zip.tmp')
    _temporary_zip.write_bytes(_payload)
    with ZipFile(_temporary_zip) as _archive:
        if _archive.comment.decode('ascii').strip() != _commit:
            raise RuntimeError('O pacote recebido não corresponde à versão publicada.')
    _temporary_zip.replace(_code_zip)

if _code_zip.is_file():
    _loaded = any(name == 'iatan' or name.startswith('iatan.') for name in sys.modules)
    _previous_record = root / 'codigo_publicado.json'
    _previous_commit = (json.loads(_previous_record.read_text(encoding='utf-8')).get('commit')
                        if _previous_record.is_file() else None)
    with ZipFile(_code_zip) as _archive, TemporaryDirectory(prefix='iatan-setup-') as _temporary:
        _helpers = [name for name in _archive.namelist() if name.endswith('/iatan/colab.py')
                    or name == 'iatan/colab.py']
        if len(_helpers) != 1:
            raise RuntimeError('O pacote de código está incompleto; atualize Iatan_codigo_atual.zip.')
        _helper_file = Path(_temporary) / 'colab_setup.py'
        _helper_file.write_bytes(_archive.read(_helpers[0]))
        _spec = importlib.util.spec_from_file_location('iatan_setup', _helper_file)
        _helper = importlib.util.module_from_spec(_spec)
        _spec.loader.exec_module(_helper)
        _record = _helper.synchronize_archive(root, _code_zip)
    if _loaded and _previous_commit != _record['commit']:
        raise RuntimeError('Código atualizado. Reinicie a sessão e execute esta célula novamente '
                           'para não misturar versões de módulos já carregados.')

# 3. PREPARAR DEPENDÊNCIAS E CONFERIR OS INSUMOS DESTA SESSÃO
sys.path.insert(0, str(root)) if str(root) not in sys.path else None
from iatan.colab import prepare_session
root = prepare_session(root, delivery=_delivery, install=_drive is not None,
                       require_multiyear=False)
os.environ['IATAN_PROJECT_ROOT'] = str(root)

# 4. ATUALIZAR AS CÓPIAS DE EXECUÇÃO E MOSTRAR ONDE CONSULTAR AS SAÍDAS
from iatan.colab import copy_notebooks
copy_notebooks(root, _delivery)
print('Notebooks atuais: Notebooks/00 a 06.')
print('Resultados de 2025: execucao/outputs/fase2_hmm_process_mining/, '
      'lag_selection_predictive/ e phase_04_predictive_modeling/.')
print('Exp33: execucao/outputs/phase_04_ablation_foldwise/.')
print('Notebook 06: execucao/outputs/phase_05_*/; cada rodada tem sua pasta própria.')


## Reinicialização e versão

Se a preparação atualizar uma biblioteca que já estava carregada, use **Ambiente de execução → Reiniciar sessão** e execute a célula novamente. Isso evita misturar versões; os arquivos de dados e os resultados não são removidos.

O commit e os hashes dos arquivos sincronizados ficam em `execucao/codigo_publicado.json`. Cada rodada registra os pacotes e os hashes realmente usados no MLflow. A rodada de referência utilizou Python 3.12; uma execução em Python 3.13 é registrada com sua versão real.
